# Notebook 1: Prepare the 2025 311 Dashboard Dataset

## Purpose
Your original 311 file contains more rows and columns than a Streamlit dashboard needs. In this notebook, you will create a **smaller 2025 deployment dataset**.

You will:
1. Read the full 311 dataset.
2. Convert the request date to a real datetime value.
3. **Keep only requests from 2025.**
4. Create a few useful dashboard variables.
5. Select only the columns your dashboard needs.
6. Save the result as `311_2025_dashboard.csv`.
7. Check the new file size before using GitHub.

> **Important:** You are expected to edit the list of columns. Do not keep a column just because it appears in the starter code.


## 1. Import pandas and locate your file
Change `FULL_DATA_FILE` so that it matches the name/path of the full 311 file you downloaded.


In [ ]:
import pandas as pd
from pathlib import Path

# EDIT THIS filename/path if needed
FULL_DATA_FILE = "311_service_requests.csv"

# We will save the smaller file with this name
OUTPUT_FILE = "311_2025_dashboard.csv"


## 2. Read the full dataset
If your source file is a CSV, use the code below. If your instructor supplied a different file type, adjust the read command as directed.


In [ ]:
df = pd.read_csv(FULL_DATA_FILE, low_memory=False)

print(f"Rows in full dataset: {len(df):,}")
print(f"Columns in full dataset: {len(df.columns):,}")
df.head()


## 3. Look at the available columns
Before deleting anything, see what is available. Your dashboard should keep only fields that support the questions and charts you want to show.


In [ ]:
print(df.columns.tolist())


## 4. Convert the request timestamp and keep ONLY 2025
The dashboard for this project must use **2025 requests only**.

`errors="coerce"` changes invalid date values to missing values instead of crashing the notebook.


In [ ]:
df["requested_datetime"] = pd.to_datetime(
    df["requested_datetime"],
    errors="coerce",
    utc=True
)

# Required: keep only 2025 requests
df_2025 = df[df["requested_datetime"].dt.year == 2025].copy()

print(f"2025 rows: {len(df_2025):,}")
print("First request:", df_2025["requested_datetime"].min())
print("Last request: ", df_2025["requested_datetime"].max())


## 5. Create variables that will be useful in the dashboard
The most important new field is `request_date`. Streamlit will later use it to let the user choose a **start date** and **end date**.

We will also create `month` and `day_of_week`. You may use these in your dashboard, or you may remove them later if you do not need them.


In [ ]:
# A date-only field is convenient for dashboard filtering
df_2025["request_date"] = df_2025["requested_datetime"].dt.date

# Optional but useful dashboard variables
df_2025["month"] = df_2025["requested_datetime"].dt.month_name()
df_2025["day_of_week"] = df_2025["requested_datetime"].dt.day_name()

print("Dashboard date range:")
print(df_2025["request_date"].min(), "through", df_2025["request_date"].max())


## 6. Optional: Calculate resolution time
If you want to analyze how long requests take to close, create `resolution_days`.

If you do **not** plan to use resolution time in your dashboard, you can skip this section.


In [ ]:
if "closed_datetime" in df_2025.columns:
    df_2025["closed_datetime"] = pd.to_datetime(
        df_2025["closed_datetime"],
        errors="coerce",
        utc=True
    )

    df_2025["resolution_days"] = (
        df_2025["closed_datetime"] - df_2025["requested_datetime"]
    ).dt.total_seconds() / 86400

    print(df_2025["resolution_days"].describe())


## 7. Choose the columns for YOUR dashboard
The list below is only a starting point. **Edit it.**

Keep fields needed for your charts, filters, metrics, or map. Remove fields you do not use. Large text/media fields such as `media_url`, `status_notes`, `subject`, or `gdb_geomattr_data` usually do not belong in this dashboard dataset unless you have a specific reason to use them.

If you created `resolution_days` and want to use it, add it to the list.


In [ ]:
# EDIT THIS LIST based on the dashboard you plan to build
columns_to_keep = [
    "service_request_id",
    "request_date",
    "requested_datetime",
    "status",
    "service_name",
    "agency_responsible",
    "zipcode",
    "lat",
    "lon",
    "month",
    "day_of_week"
]

# Keep only columns that actually exist in the source file.
# This makes the starter notebook a little more forgiving if column names vary.
columns_to_keep = [c for c in columns_to_keep if c in df_2025.columns]

dashboard_df = df_2025[columns_to_keep].copy()

print(f"Dashboard rows: {len(dashboard_df):,}")
print(f"Dashboard columns: {len(dashboard_df.columns)}")
dashboard_df.head()


## 8. Check missing values and data types
Do a quick check before saving. This is not another full EDA; you are simply making sure the deployment data looks reasonable.


In [ ]:
print(dashboard_df.dtypes)
print("
Missing values:")
print(dashboard_df.isna().sum().sort_values(ascending=False))


## 9. Save the reduced dataset
This is the file that your second notebook and your Streamlit application will use.


In [ ]:
dashboard_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")


## 10. Check the saved file size
For this assignment, keep the dashboard data file **under 25 MB** so it is easy to work with in GitHub.

If the file is still too large, go back to your `columns_to_keep` list and remove fields your dashboard does not need. You may also discuss other reasonable reductions with your instructor.


In [ ]:
file_size_mb = Path(OUTPUT_FILE).stat().st_size / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

if file_size_mb < 25:
    print("Good: the file is under 25 MB.")
else:
    print("The file is still too large for this assignment. Reduce it further.")


## 11. Final verification
Your reduced dataset should:
- contain **2025 only**;
- include `request_date`;
- include only fields that have a purpose in your dashboard;
- be under **25 MB**;
- be saved as `311_2025_dashboard.csv`.

You are now ready for **Notebook 2: Prototype Your Dashboard Visualizations**.
